<div dir="rtl">
<h1>پیام مشترک، پردازش جداگانهٔ هر موقعیت</h1>
<p>درس 43 از 76 · بعد از ارتباط، روی ویژگی‌ها چه محاسبه‌ای کنیم؟ · <code dir="ltr">37-ffn</code></p>
<p><a target="_self" href="http://127.0.0.1:8000/part-06/chapter-02/37-ffn.html">📖 بازگشت به همین درس</a></p>
<p>FFN واقعی را از وزن‌هایش بازسازی کنید و استقلال موقعیت‌ها را بسنجید.</p><p>پیش‌نیاز: دو Linear با Activation میان آن‌ها و محور آخر ویژگی را بشناسید.</p>
<p>این دفتر نیمهٔ عملی درس است. مثال‌ها آمادهٔ اجرا هستند؛ دو Cell با برچسب TODO را خودتان کامل کنید. پیام INCOMPLETE یعنی هنوز چیزی ننوشته‌اید، نه اینکه پاسخ درست است. جواب مرجع در این دفتر پنهان نشده است.</p>
<p>از بالا به پایین اجرا کنید. پس از تغییر هر تابع، Cell آن و سپس Cell آزمون را دوباره اجرا کنید. برای بررسی نهایی، از منوی <code>Kernel → Restart Kernel and Run All Cells</code> استفاده کنید.</p>
</div>

In [ ]:
from pathlib import Path
import os
import sys

project_root = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "mini_gpt").is_dir() and (p / "book_src").is_dir()), None)
if project_root is None:
    raise RuntimeError("Extract the complete learning project; open this notebook inside it.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
print("Python:", sys.executable)
print("Project:", project_root)

<div dir="rtl">
<h2>قبل از اجرا، پیش‌بینی کنید</h2>
<p>اگر دو موقعیت ورودی یکسان باشند، FFN با وزن مشترک چه خروجی‌هایی می‌دهد؟ تغییر موقعیت سوم به آن دو راهی دارد؟</p>
</div>

<div dir="rtl"><p>پیش‌بینی من: …</p></div>

In [ ]:
import math
import torch
torch.set_num_threads(1)
torch.manual_seed(17)
from torch.nn import functional as F
from mini_gpt.config import ModelConfig
from mini_gpt.transformer import FeedForward
ffn = FeedForward(ModelConfig(12,8,4,1,1,0.)).eval()
x = torch.randn(2,3,4)
x[:,1] = x[:,0]
print('matching positions:',torch.equal(x[:,0],x[:,1]))

<div dir="rtl">
<h2>این بار شما کد بنویسید</h2>
<p>تابع manual_ffn(Feed-Forward Network, x) را با وزن و Bias Layer‌های ۰ و ۲ در ffn.layers بنویسید. بین دو تبدیل F.gelu به کار ببرید؛ خود ffn(x) یا Sequential کامل را صدا نزنید. Dropout این آزمایش صفر است.</p>
</div>

In [ ]:
def manual_ffn(ffn, x):
    # TODO
    return None

In [ ]:
def test_exercise():
    result = manual_ffn(ffn,x)
    if result is None: return False
    torch.testing.assert_close(result,ffn(x))
    torch.testing.assert_close(result[:,0],result[:,1])
    changed = x.clone(); changed[:,2] += 20
    torch.testing.assert_close(manual_ffn(ffn,changed)[:,:2],result[:,:2])
    other = torch.randn(1,5,4)
    torch.testing.assert_close(manual_ffn(ffn,other),ffn(other))
    return True

exercise_complete = test_exercise()
print("PASS" if exercise_complete else "INCOMPLETE: complete the TODO first")

<div dir="rtl">
<h2>فقط یک عامل را تغییر دهید</h2>
<p>فقط موقعیت آخر را به اندازهٔ ۱۰ افزایش دهید. اختلاف خروجی هر موقعیت را جدا چاپ کنید؛ صفرهای سایر موقعیت‌ها مهم‌تر از مقدار عددی اختلاف آخرند.</p>
</div>

In [ ]:
changed = x.clone(); changed[:,-1] += 10
with torch.no_grad():
    print('per-position change:',(ffn(changed)-ffn(x)).abs().amax(-1))

<div dir="rtl">
<h2>خرابی را پیدا کنید</h2>
<p>نسخهٔ خراب قبل از FFN میانگین زمان را می‌گیرد و به همهٔ موقعیت‌ها می‌دهد. تابع local_ffn(Feed-Forward Network,x) را اصلاح کنید؛ در این بخش می‌توانید خود Feed-Forward Network را فراخوانی کنید.</p>
</div>

In [ ]:
with torch.no_grad():
    wrong = ffn(x.mean(1,keepdim=True)).expand_as(x)
print('wrong: all positions identical:',torch.equal(wrong[:,0],wrong[:,2]))

<div dir="rtl">
<h2>اصلاح را خودتان بنویسید</h2>
<p>علت را توضیح دهید، سپس تابع زیر را کامل کنید. خطای عمدی بالا یک نمونهٔ آموزشی است؛ آزمون پایین باید اصلاح شما را بسنجد.</p>
</div>

In [ ]:
def local_ffn(ffn, x):
    # TODO
    return None

In [ ]:
def test_repair():
    result = local_ffn(ffn,x)
    if result is None: return False
    torch.testing.assert_close(result,ffn(x))
    changed = x.clone(); changed[:,2] -= 100
    torch.testing.assert_close(local_ffn(ffn,changed)[:,:2],result[:,:2])
    return True

repair_complete = test_repair()
print("PASS" if repair_complete else "INCOMPLETE: complete the TODO first")

<div dir="rtl">
<h2>در Mini-GPT کجا به کار می‌آید؟</h2>
<p>این همان FeedForward در mini_gpt/transformer.py است. FFN اطلاعات همسایه را مستقیماً نمی‌خواند؛ اطلاعات گذشته پیش‌تر از راه Attention وارد نمایش آن موقعیت شده است.</p>
</div>

<div dir="rtl">
<h2>با زبان خودتان توضیح دهید</h2>
<p>اگر FFN را تنها نگه داریم، تغییر گذشته از کدام مسیر می‌تواند به Token فعلی برسد؟ چرا وزن مشترک با ورودی مشترک یکی نیست؟</p>
</div>
<div dir="rtl"><p>پیش‌بینی و مشاهدهٔ من: …</p><p>علت خرابی و اصلاح من: …</p></div>

<div dir="rtl"><p><a target="_self" href="http://127.0.0.1:8000/part-06/chapter-02/37-ffn.html">بازگشت به درس و ادامهٔ مسیر</a> · <a target="_self" href="http://127.0.0.1:8000/answers/37-ffn.html#lab-solution">فقط پس از تلاش: راه‌حل مرجع آزمایشگاه</a></p></div>